# 1. Experiment Overview

This notebook implements the experimental pipeline used to compare tree-based machine learning algorithms for multi-class classification under different dataset characteristics.

The experiments analyze the impact of:

- dataset size
- class imbalance

The following algorithms are evaluated:

- Decision Tree (CART)
- Random Forest
- Extra Trees
- Gradient Boosting
- XGBoost
- LightGBM
- CatBoost

The pipeline performs the following steps:

1. dataset loading and preprocessing  
2. dataset size and class imbalance generation  
3. hyperparameter optimization  
4. model training and evaluation  
5. statistical significance testing  
6. result aggregation and visualization

## 2. Project Structure and Execution

This notebook runs the experimental pipeline for the multi-class classification part of the study and interacts with the GitHub repository that stores datasets, tuned hyperparameters, and previously generated results.

When executed in Google Colab, the repository is cloned so that the notebook can access the required datasets and configuration files.

The repository contains:

- `datasets/original_datasets`  
  Original datasets used in the experiments.

- `datasets/generated_datasets`  
  Generated dataset subsets for dataset size and class imbalance experiments.

- `config`  
  Tuned hyperparameters stored as JSON files.

- `results`  
  Experimental results generated by the notebook.

- `results/figures`  
  Visualizations produced from experiment results.

During execution, the notebook generates intermediate datasets, results, and figures in the runtime environment. These files can then be downloaded and uploaded to the repository if needed.

In [1]:

# Repository Setup


import os

REPO_NAME = "Tree-algorithms-dataset-characteristics"
REPO_URL = "https://github.com/Ilaha-Habibova/Tree-algorithms-dataset-characteristics.git"

if "COLAB_GPU" in os.environ and not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}

if "COLAB_GPU" in os.environ and os.path.exists(REPO_NAME):
    %cd {REPO_NAME}

# Paths used in the notebook

BASE_PATH = os.getcwd()

# Dataset location
ORIGINAL_DATASETS_PATH = os.path.join(
    BASE_PATH, "datasets", "original_datasets"
)

# Hyperparameter configuration files
CONFIG_PATH = os.path.join(
    BASE_PATH, "config"
)

print("Repository ready.")


Cloning into 'Tree-algorithms-dataset-characteristics'...
remote: Enumerating objects: 447, done.
remote: Counting objects: 100% (117/117), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 447 (delta 82), reused 40 (delta 40), pack-reused 330 (from 2)
Receiving objects: 100% (447/447), 20.33 MiB | 20.73 MiB/s, done.
Resolving deltas: 100% (181/181), done.
/content/Tree-algorithms-dataset-characteristics
Repository ready.


## 3. Environment Setup and Library Versions

This section installs the required libraries and imports all packages used throughout the experimental pipeline.

The libraries support:

- data manipulation and numerical computation
- implementation of machine learning algorithms
- statistical significance testing
- visualization of experimental results

For reproducibility, the versions of the main libraries used in the experiment are also recorded.

In [2]:
# Install required libraries

!pip install -q catboost scikit-posthocs itables requests


# ALL IMPORTS
import os
import json
import time
import random
import warnings
import pickle
import requests
import shutil

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
import scikit_posthocs as sp

from sklearn.model_selection import (
    StratifiedKFold,
    StratifiedShuffleSplit,
    RandomizedSearchCV,
    cross_validate
)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.base import clone

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier
)

import xgboost as xgb
import lightgbm as lgb
import catboost as cb

warnings.filterwarnings("ignore")


# Random Seed Configuration
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

print("Random seed set to:", SEED)


# GLOBAL CV
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

# Model Size Utility
def get_model_size_kb(model):
    return len(pickle.dumps(model)) / 1024

# Library versions (reproducibility)
import sklearn
import scipy
import matplotlib
import scikit_posthocs

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("xgboost:", xgb.__version__)
print("lightgbm:", lgb.__version__)
print("catboost:", cb.__version__)
print("scipy:", scipy.__version__)
print("matplotlib:", matplotlib.__version__)
print("seaborn:", sns.__version__)
print("scikit-posthocs:", scikit_posthocs.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 28.9 MB/s eta 0:00:00
Random seed set to: 42
numpy: 2.0.2
pandas: 2.2.2
scikit-learn: 1.6.1
xgboost: 3.2.0
lightgbm: 4.6.0
catboost: 1.2.10
scipy: 1.16.3
matplotlib: 3.10.0
seaborn: 0.13.2
scikit-posthocs: 0.13.0


## 4. Dataset Loading and Inspection

This section loads the original multi-class classification datasets used in the experiments and performs a basic inspection of their structure.

The following datasets are used:

- **Dry Bean Dataset**
- **Online Retail Customer Segmentation Dataset**

After loading the datasets, a brief inspection is performed to verify:

- dataset dimensions
- feature data types
- missing values
- class distributions

In [3]:
# Dataset Loading

# File paths
beans_path = os.path.join(
    ORIGINAL_DATASETS_PATH,
    "Dry_Bean_Dataset.csv"
)

retail_path = os.path.join(
    ORIGINAL_DATASETS_PATH,
    "retail_customer_segmentation.csv"
)

# Load datasets
beans = pd.read_csv(beans_path)
retail = pd.read_csv(retail_path)

# Preview
print("Dry Bean dataset preview:")
display(beans.head())

print("\nRetail dataset preview:")
display(retail.head())

Dry Bean dataset preview:


,Area,Perimeter,MajorAxisLength,MinorAxisLength,AspectRation,Eccentricity,ConvexArea,EquivDiameter,Extent,Solidity,roundness,Compactness,ShapeFactor1,ShapeFactor2,ShapeFactor3,ShapeFactor4,Class
0,28395,610.291,208.178117,173.888747,1.197191,0.549812,28715,190.141097,0.763923,0.988856,0.958027,0.913358,0.007332,0.003147,0.834222,0.998724,SEKER
1,28734,638.018,200.524796,182.734419,1.097356,0.411785,29172,191.272751,0.783968,0.984986,0.887034,0.953861,0.006979,0.003564,0.909851,0.998430,SEKER
2,29380,624.110,212.826130,175.931143,1.209713,0.562727,29690,193.410904,0.778113,0.989559,0.947849,0.908774,0.007244,0.003048,0.825871,0.999066,SEKER
3,30008,645.884,210.557999,182.516516,1.153638,0.498616,30724,195.467062,0.782681,0.976696,0.903936,0.928329,0.007017,0.003215,0.861794,0.994199,SEKER
4,30140,620.134,201.847882,190.279279,1.060798,0.333680,30417,195.896503,0.773098,0.990893,0.984877,0.970516,0.006697,0.003665,0.941900,0.999166,SEKER



Retail dataset preview:


,customer_id,age,annual_income,months_active,avg_monthly_spend,purchase_frequency,avg_order_value,discount_usage_rate,return_rate,browsing_time_minutes,support_interactions,payment_method,region,customer_segment
0,33554,53,100473.211709,63,121.430565,0.817268,66.820403,0.117256,0.023144,77.298393,2.0,Card,Semi-Urban,Occasional
1,9428,54,54730.644845,67,572.552674,3.176551,137.087449,0.261647,0.429054,92.132565,2.0,Wallet,Urban,Occasional
2,200,44,58268.121079,57,266.593896,2.713168,71.796888,0.284785,0.011854,155.194768,1.0,UPI,Rural,Occasional
3,12448,54,64829.795654,40,691.452358,5.553977,105.501185,0.104832,0.399686,113.917756,0.0,Wallet,Rural,High_Value
4,39490,28,27431.467873,15,832.664792,1.348389,354.568534,0.409204,0.039517,50.123656,1.0,Card,Semi-Urban,Occasional


In [4]:
# Dataset Inspection

def inspect_dataset(df, target, name):

    print("\n====================================")
    print(f"{name.upper()} DATASET")
    print("====================================")

    print("\nShape:", df.shape)

    print("\nFeature types:")
    print(df.dtypes)

    print("\nMissing values:")
    print(df.isnull().sum())

    print(f"\nTarget distribution ({target}):")
    print(df[target].value_counts())

    print("\nClass proportions:")
    print(df[target].value_counts(normalize=True))

# Inspect

inspect_dataset(beans, "Class", "Dry Bean")

inspect_dataset(retail, "customer_segment", "Retail")


DRY BEAN DATASET

Shape: (13611, 17)

Feature types:
Area                 int64
Perimeter          float64
MajorAxisLength    float64
MinorAxisLength    float64
AspectRation       float64
Eccentricity       float64
ConvexArea           int64
EquivDiameter      float64
Extent             float64
Solidity           float64
roundness          float64
Compactness        float64
ShapeFactor1       float64
ShapeFactor2       float64
ShapeFactor3       float64
ShapeFactor4       float64
Class               object
dtype: object

Missing values:
Area               0
Perimeter          0
MajorAxisLength    0
MinorAxisLength    0
AspectRation       0
Eccentricity       0
ConvexArea         0
EquivDiameter      0
Extent             0
Solidity           0
roundness          0
Compactness        0
ShapeFactor1       0
ShapeFactor2       0
ShapeFactor3       0
ShapeFactor4       0
Class              0
dtype: int64

Target distribution (Class):
Class
DERMASON    3546
SIRA        2636
SEKER       2027

In [5]:
# Drop rows with ANY missing values for Retail Dataset

retail = retail.dropna().reset_index(drop=True)

print("After dropna():", retail.shape)

print("\n=== AFTER DROPPING MISSING VALUES ===")
print(retail["customer_segment"].value_counts())
print(retail["customer_segment"].value_counts(normalize=True))

After dropna(): (34139, 14)

=== AFTER DROPPING MISSING VALUES ===
customer_segment
Occasional    15105
Regular        9215
Loyal          6221
High_Value     3598
Name: count, dtype: int64
customer_segment
Occasional    0.442456
Regular       0.269926
Loyal         0.182226
High_Value    0.105393
Name: proportion, dtype: float64


## 5. Feature and Target Definition

In this section, the predictor variables (**X**) and target variables (**y**) are defined for both multi-class datasets.

For the **Dry Bean dataset**, the target variable `Class` contains textual labels representing bean varieties. These labels are converted into integer values so that the algorithms can perform multi-class classification.

For the **Online Retail dataset**, the target variable `customer_segment` contains categorical customer segment labels and is also converted into integer values.

After defining the feature matrices and target variables, feature types are identified. Categorical variables are detected based on their data types and are later transformed using one-hot encoding, while numerical variables are passed directly to the models.

In [6]:

# Feature and Target Definition (MULTI-CLASS DATASETS)

# Dry Bean dataset


bean_label_encoder = LabelEncoder()

y_beans_encoded = bean_label_encoder.fit_transform(beans["Class"])

y_beans = pd.Series(
    y_beans_encoded,
    name="Class"
)

X_beans = beans.drop(columns=["Class"])

# Retail dataset

# Drop ID column
retail = retail.drop(columns=["customer_id"], errors="ignore")

# Encode target
retail_label_encoder = LabelEncoder()

y_retail_encoded = retail_label_encoder.fit_transform(
    retail["customer_segment"]
)

y_retail = pd.Series(
    y_retail_encoded,
    name="customer_segment"
)

X_retail = retail.drop(columns=["customer_segment"])



# Save label mappings

print("\nDry Bean Label Mapping:")
for k, v in zip(
    bean_label_encoder.classes_,
    bean_label_encoder.transform(bean_label_encoder.classes_)
):
    print(f"{k} → {v}")

print("\nRetail Label Mapping:")
for k, v in zip(
    retail_label_encoder.classes_,
    retail_label_encoder.transform(retail_label_encoder.classes_)
):
    print(f"{k} → {v}")

# Feature type identification

# Dry Bean (all numerical)
categorical_features_beans = []
numerical_features_beans = X_beans.columns.tolist()


# Retail
categorical_features_retail = X_retail.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numerical_features_retail = X_retail.select_dtypes(
    exclude=["object", "category"]
).columns.tolist()


# Verification

print("\nDry Bean feature matrix:", X_beans.shape)
print("Dry Bean target:", y_beans.shape)

print("\nRetail feature matrix:", X_retail.shape)
print("Retail target:", y_retail.shape)

print("\nDry Bean numerical features:", numerical_features_beans)

print("\nRetail categorical features:", categorical_features_retail)
print("Retail numerical features:", numerical_features_retail)


Dry Bean Label Mapping:
BARBUNYA → 0
BOMBAY → 1
CALI → 2
DERMASON → 3
HOROZ → 4
SEKER → 5
SIRA → 6

Retail Label Mapping:
High_Value → 0
Loyal → 1
Occasional → 2
Regular → 3

Dry Bean feature matrix: (13611, 16)
Dry Bean target: (13611,)

Retail feature matrix: (34139, 12)
Retail target: (34139,)

Dry Bean numerical features: ['Area', 'Perimeter', 'MajorAxisLength', 'MinorAxisLength', 'AspectRation', 'Eccentricity', 'ConvexArea', 'EquivDiameter', 'Extent', 'Solidity', 'roundness', 'Compactness', 'ShapeFactor1', 'ShapeFactor2', 'ShapeFactor3', 'ShapeFactor4']

Retail categorical features: ['payment_method', 'region']
Retail numerical features: ['age', 'annual_income', 'months_active', 'avg_monthly_spend', 'purchase_frequency', 'avg_order_value', 'discount_usage_rate', 'return_rate', 'browsing_time_minutes', 'support_interactions']


## 6. Preprocessing Pipeline

Before training the models, input features must be transformed into a format suitable for machine learning algorithms.

The **Dry Bean dataset** contains only numerical features, so no categorical encoding is required.

The **Online Retail dataset** contains both numerical and categorical features. Categorical variables are converted into numerical representations using one-hot encoding, while numerical variables are passed directly to the model.

Preprocessing is implemented using **scikit-learn’s ColumnTransformer**, which allows different transformations to be applied to different feature groups. Integrating preprocessing into a pipeline ensures that transformations are applied consistently during cross-validation and prevents data leakage.

In [7]:
# Preprocessing Pipeline
# Common categorical transformer
categorical_transformer = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

# Dry Bean dataset
# All features are numerical

preprocessor_beans = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numerical_features_beans)
    ]
)

print("Dry Bean preprocessing defined.")

# Retail dataset
# Mixed: categorical + numerical

preprocessor_retail = ColumnTransformer(
    transformers=[
        ("cat", categorical_transformer, categorical_features_retail),
        ("num", "passthrough", numerical_features_retail)
    ]
)

print("Retail preprocessing defined.")

Dry Bean preprocessing defined.
Retail preprocessing defined.


## 7. Model Definitions

This section initializes the machine learning algorithms evaluated in the experiments.

At this stage, only model objects are created. The models will later be combined with the preprocessing pipeline and used during hyperparameter optimization and evaluation.

All models are stored in a dictionary, where the key represents the algorithm name and the value is the corresponding model object. This structure allows the experiment to iterate automatically over all algorithms under identical experimental conditions.

In [8]:
# Model Definitions

models = {

    # Decision Tree (CART)
    "DecisionTree": DecisionTreeClassifier(
        random_state=SEED
    ),

    # Random Forest
    "RandomForest": RandomForestClassifier(
        random_state=SEED,
        n_jobs=1
    ),

    # Extra Trees
    "ExtraTrees": ExtraTreesClassifier(
        random_state=SEED,
        n_jobs=1
    ),

    # Gradient Boosting
    "GradientBoosting": GradientBoostingClassifier(
        random_state=SEED
    ),

    # XGBoost
    "XGBoost": xgb.XGBClassifier(
        random_state=SEED,
        n_jobs=1,
        verbosity=0,
        eval_metric="logloss"
    ),

    # LightGBM
    "LightGBM": lgb.LGBMClassifier(
        random_state=SEED,
        n_jobs=1,
        verbosity=-1
    ),

    # CatBoost
    "CatBoost": cb.CatBoostClassifier(
        random_state=SEED,
        verbose=0,
        thread_count=1
    )
}

print("Models initialized:", list(models.keys()))

Models initialized: ['DecisionTree', 'RandomForest', 'ExtraTrees', 'GradientBoosting', 'XGBoost', 'LightGBM', 'CatBoost']


## 8. Hyperparameter Optimization

Hyperparameters are optimized using **Randomized Search with stratified five-fold cross-validation**.

The tuning procedure is performed once for each original multi-class dataset. If previously tuned hyperparameters are already available in the repository, they are loaded directly in order to avoid repeating the computationally expensive tuning stage.

**Macro F1-score** is used as the optimization metric because it gives equal importance to all classes and is consistent with the primary evaluation metric used in the multi-class experiments.

In [9]:
# Hyperparameter Optimization

import json
import requests
import warnings
warnings.filterwarnings("ignore")

# Hyperparameter search spaces

param_distributions = {

    "DecisionTree": {
        "model__max_depth": [None, 5, 10, 20, 30],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 4]
    },

    "RandomForest": {
        "model__n_estimators": [100, 200, 300],
        "model__max_depth": [None, 10, 20, 30],
        "model__max_features": ["sqrt", "log2"],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 4]
    },

    "ExtraTrees": {
        "model__n_estimators": [100, 200, 300],
        "model__max_depth": [None, 10, 20, 30],
        "model__max_features": ["sqrt", "log2"],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 4]
    },

    "GradientBoosting": {
        "model__n_estimators": [100, 200, 300],
        "model__learning_rate": [0.01, 0.05, 0.1],
        "model__subsample": [0.8, 1.0],
        "model__max_depth": [3, 5, 7],
        "model__max_features": ["sqrt", "log2"],
        "model__min_samples_split": [2, 5]
    },

    "XGBoost": {
        "model__n_estimators": [100, 200, 300],
        "model__learning_rate": [0.01, 0.05, 0.1],
        "model__subsample": [0.8, 1.0],
        "model__colsample_bytree": [0.8, 1.0],
        "model__max_depth": [3, 5, 7]
    },

    "LightGBM": {
        "model__n_estimators": [100, 200, 300],
        "model__learning_rate": [0.01, 0.05, 0.1],
        "model__num_leaves": [31, 50, 70],
        "model__feature_fraction": [0.8, 1.0]
    },

    "CatBoost": {
        "model__iterations": [200, 400, 600],
        "model__learning_rate": [0.01, 0.05, 0.1],
        "model__depth": [4, 6, 8],
        "model__l2_leaf_reg": [1, 3, 5]
    }
}

# Tuning function (Macro-F1)

def tune_models(X, y, preprocessor):

    best_params = {}

    for model_name, model in models.items():

        print(f"Tuning {model_name}...")

        pipeline = Pipeline([
            ("preprocessing", preprocessor),
            ("model", model)
        ])

        search = RandomizedSearchCV(
            estimator=pipeline,
            param_distributions=param_distributions[model_name],
            n_iter=10,
            cv=cv,
            scoring="f1_macro",
            n_jobs=1,
            random_state=SEED,
            verbose=0
        )

        search.fit(X, y)

        best_params[model_name] = {
            "best_params": search.best_params_,
            "best_score": search.best_score_
        }

        print("Best score:", round(search.best_score_, 4))

    return best_params

# GitHub URLs

BEANS_URL = "https://raw.githubusercontent.com/Ilaha-Habibova/Tree-algorithms-dataset-characteristics/main/config/best_params_beans_f1.json"
RETAIL_URL = "https://raw.githubusercontent.com/Ilaha-Habibova/Tree-algorithms-dataset-characteristics/main/config/best_params_retail_f1.json"

# Local paths

beans_params_path = os.path.join(CONFIG_PATH, "best_params_beans_f1.json")
retail_params_path = os.path.join(CONFIG_PATH, "best_params_retail_f1.json")

os.makedirs(CONFIG_PATH, exist_ok=True)


# Load function

def load_from_github(url):
    try:
        r = requests.get(url)
        if r.status_code == 200:
            print(f"🌐 Loaded from GitHub: {url}")
            return r.json()
        else:
            print(f"❌ GitHub file not found: {url}")
            return None
    except Exception as e:
        print(f"⚠️ GitHub load failed: {e}")
        return None

# Load or tune
tuned_now = False

best_params_beans = load_from_github(BEANS_URL)
best_params_retail = load_from_github(RETAIL_URL)

# Fallback: local
if best_params_beans is None and os.path.exists(beans_params_path):
    print("📂 Loading beans params from local...")
    with open(beans_params_path) as f:
        best_params_beans = json.load(f)

if best_params_retail is None and os.path.exists(retail_params_path):
    print("📂 Loading retail params from local...")
    with open(retail_params_path) as f:
        best_params_retail = json.load(f)

# Run tuning if missing
if best_params_beans is None or best_params_retail is None:

    print("\n🚀 Running hyperparameter tuning (Macro-F1)...")

    if best_params_beans is None:
        best_params_beans = tune_models(
            X_beans,
            y_beans,
            preprocessor_beans
        )
        with open(beans_params_path, "w") as f:
            json.dump(best_params_beans, f, indent=4)
        tuned_now = True

    if best_params_retail is None:
        best_params_retail = tune_models(
            X_retail,
            y_retail,
            preprocessor_retail
        )
        with open(retail_params_path, "w") as f:
            json.dump(best_params_retail, f, indent=4)
        tuned_now = True

    print("\n💾 Multi-class hyperparameters tuned and saved.")

else:
    print("\n✅ Multi-class hyperparameters loaded (GitHub/local).")


# Download ONLY if tuning was executed
if tuned_now:
    try:
        from google.colab import files
        print("\n⬇️ Downloading newly tuned hyperparameters...")
        files.download(beans_params_path)
        files.download(retail_params_path)
    except:
        print("Download skipped (not in Colab).")
else:
    print("\n📁 No download needed (loaded from GitHub/local).")

🌐 Loaded from GitHub: https://raw.githubusercontent.com/Ilaha-Habibova/Tree-algorithms-dataset-characteristics/main/config/best_params_beans_f1.json
🌐 Loaded from GitHub: https://raw.githubusercontent.com/Ilaha-Habibova/Tree-algorithms-dataset-characteristics/main/config/best_params_retail_f1.json

✅ Multi-class hyperparameters loaded (GitHub/local).

📁 No download needed (loaded from GitHub/local).


## 9. Dataset Size Experiment

This experiment evaluates how classifier performance changes as dataset size increases.

Three dataset sizes are considered:

- Small: 1,000 observations
- Medium: 3,000 observations
- Large: 9,000 observations

Dataset subsets are generated using **nested stratified sampling** in order to preserve class distribution across all size levels.

To improve computational efficiency and reproducibility:

- generated subsets are stored as CSV files
- if the files already exist, they are loaded instead of regenerated

Model evaluation is later performed using stratified five-fold cross-validation with previously optimized hyperparameters.

In [10]:

# DATASET SIZE — MULTI-CLASS

from sklearn.model_selection import StratifiedShuffleSplit
import os, shutil, requests, zipfile
import numpy as np
import pandas as pd

size_levels = [1000, 3000, 9000]

SIZE_DATASETS_PATH = os.path.join(
    BASE_PATH,
    "datasets",
    "generated_datasets",
    "size_experiments"
)
os.makedirs(SIZE_DATASETS_PATH, exist_ok=True)

# File paths
beans_files = {
    s: os.path.join(SIZE_DATASETS_PATH, f"beans_size_{s}.csv")
    for s in size_levels
}

retail_files = {
    s: os.path.join(SIZE_DATASETS_PATH, f"retail_size_{s}.csv")
    for s in size_levels
}

# Nested subset function
def create_nested_subsets(X, y, sizes, seed=SEED):

    X = pd.DataFrame(X).reset_index(drop=True)
    y = pd.Series(y).reset_index(drop=True)

    subsets = {}
    selected = np.array([], dtype=int)

    for size in sorted(sizes):

        if len(selected) == 0:
            splitter = StratifiedShuffleSplit(
                n_splits=1,
                train_size=size,
                random_state=seed
            )
            idx, _ = next(splitter.split(X, y))
            selected = idx

        else:
            remaining = np.setdiff1d(np.arange(len(X)), selected)

            X_rem = X.iloc[remaining]
            y_rem = y.iloc[remaining]

            needed = size - len(selected)

            splitter = StratifiedShuffleSplit(
                n_splits=1,
                train_size=needed,
                random_state=seed
            )
            add_idx, _ = next(splitter.split(X_rem, y_rem))
            add_idx = remaining[add_idx]

            selected = np.concatenate([selected, add_idx])

        subsets[size] = (
            X.iloc[selected].copy().reset_index(drop=True),
            y.iloc[selected].copy().reset_index(drop=True)
        )

    return subsets

# GitHub ZIP
ZIP_URL = "https://raw.githubusercontent.com/Ilaha-Habibova/Tree-algorithms-dataset-characteristics/main/datasets/generated_datasets/size_experiments/multiclass_size_subsets.zip"

zip_path = os.path.join(SIZE_DATASETS_PATH, "multiclass_size_subsets.zip")

print("🌐 Checking GitHub...")

github_ok = False

try:
    r = requests.get(ZIP_URL)

    if r.status_code == 200:
        with open(zip_path, "wb") as f:
            f.write(r.content)

        shutil.unpack_archive(zip_path, SIZE_DATASETS_PATH)

        github_ok = True
        print("✅ Loaded from GitHub")
        print(f"   Source: {ZIP_URL}")

    else:
        print("❌ GitHub not available")

except:
    print("⚠️ GitHub error")


# Check files

all_files = list(beans_files.values()) + list(retail_files.values())
files_exist = all(os.path.exists(f) for f in all_files)

# LOAD

if files_exist:

    beans_size_subsets = {}
    retail_size_subsets = {}

    for s in size_levels:

        df = pd.read_csv(beans_files[s])
        beans_size_subsets[s] = (
            df.drop(columns=["Class"]),
            df["Class"]
        )

        df = pd.read_csv(retail_files[s])
        retail_size_subsets[s] = (
            df.drop(columns=["customer_segment"]),
            df["customer_segment"]
        )

    source = "GitHub" if github_ok else "Local"
    print(f"📂 Loaded subsets ({source})")

# GENERATE

else:

    print("🚀 Generating subsets...")

    beans_size_subsets = create_nested_subsets(
        X_beans, y_beans, size_levels, SEED
    )

    retail_size_subsets = create_nested_subsets(
        X_retail, y_retail, size_levels, SEED
    )

    for s in size_levels:

        Xb, yb = beans_size_subsets[s]
        pd.concat([Xb, yb], axis=1).to_csv(beans_files[s], index=False)

        Xr, yr = retail_size_subsets[s]
        pd.concat([Xr, yr], axis=1).to_csv(retail_files[s], index=False)

    print("📂 Generated locally")

    zip_file = os.path.join(BASE_PATH, "multiclass_size_subsets.zip")

    with zipfile.ZipFile(zip_file, 'w') as z:
        for s in size_levels:
            z.write(beans_files[s], os.path.basename(beans_files[s]))
            z.write(retail_files[s], os.path.basename(retail_files[s]))

    print("📦 ZIP created")

    try:
        from google.colab import files
        files.download(zip_file)
    except:
        pass

🌐 Checking GitHub...
✅ Loaded from GitHub
   Source: https://raw.githubusercontent.com/Ilaha-Habibova/Tree-algorithms-dataset-characteristics/main/datasets/generated_datasets/size_experiments/multiclass_size_subsets.zip
📂 Loaded subsets (GitHub)


## 10. Dataset Size Experiment – Model Evaluation

This section evaluates model performance on the generated dataset size subsets.

For each dataset size condition, all algorithms are evaluated using stratified five-fold cross-validation. The best hyperparameters obtained during the tuning stage are reused without further optimization.

The primary evaluation metric is **Macro F1-score**.

Additional metrics are also reported:

- macro precision
- macro recall
- balanced accuracy
- training time
- model size

The results are stored in tabular form and sorted by Macro F1-score so that the best-performing algorithms can be identified clearly for each dataset size condition.

In [11]:

# DATASET SIZE RESULTS

import os
import requests
import pandas as pd
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline
from sklearn.base import clone

RESULTS_PATH = os.path.join(BASE_PATH, "results")
os.makedirs(RESULTS_PATH, exist_ok=True)

size_RESULTS_URL = "https://raw.githubusercontent.com/Ilaha-Habibova/Tree-algorithms-dataset-characteristics/main/results/dataset_size_results_multiclass.csv"

size_results_path = os.path.join(
    RESULTS_PATH,
    "dataset_size_results_multiclass.csv"
)

print("🌐 Checking GitHub for multiclass dataset size results...")

results_size_multi_df = None
ran_now = False


# STEP 1 — GitHub

try:
    r = requests.get(size_RESULTS_URL)

    if r.status_code == 200:
        with open(size_results_path, "wb") as f:
            f.write(r.content)

        results_size_multi_df = pd.read_csv(size_results_path)

        print("✅ Loaded from GitHub")
        print(f"   Source: {size_RESULTS_URL}")

    else:
        print("❌ GitHub not available")

except:
    print("⚠️ GitHub error")


# STEP 2 — Local

if results_size_multi_df is None and os.path.exists(size_results_path):
    print("📂 Loading multiclass results from local...")
    results_size_multi_df = pd.read_csv(size_results_path)


# STEP 3 — Run experiment if missing

if results_size_multi_df is None:

    print("🚀 Running multiclass dataset size experiment...")

    results_multi = []

    def evaluate_multi(subsets, dataset_name, best_params, preprocessor):

        for size in sorted(subsets.keys()):

            X_subset, y_subset = subsets[size]

            for model_name, base_model in models.items():

                model = clone(base_model)

                if model_name in best_params:
                    tuned_params = {
                        k.replace("model__", ""): v
                        for k, v in best_params[model_name]["best_params"].items()
                    }
                    model.set_params(**tuned_params)

                pipeline = Pipeline([
                    ("preprocessing", preprocessor),
                    ("model", model)
                ])

                scoring = {
                    "balanced_accuracy": "balanced_accuracy",
                    "macro_f1": "f1_macro",
                    "macro_precision": "precision_macro",
                    "macro_recall": "recall_macro"
                }

                cv_results = cross_validate(
                    pipeline,
                    X_subset,
                    y_subset,
                    cv=cv,
                    scoring=scoring,
                    n_jobs=1
                )

                pipeline.fit(X_subset, y_subset)

                results_multi.append({
                    "dataset": dataset_name,
                    "size": size,
                    "model": model_name,
                    "balanced_accuracy": cv_results["test_balanced_accuracy"].mean(),
                    "macro_f1": cv_results["test_macro_f1"].mean(),
                    "macro_precision": cv_results["test_macro_precision"].mean(),
                    "macro_recall": cv_results["test_macro_recall"].mean(),
                    "training_time": cv_results["fit_time"].mean(),
                    "model_size_kb": get_model_size_kb(pipeline.named_steps["model"])
                })

    evaluate_multi(
        beans_size_subsets,
        "DryBean",
        best_params_beans,
        preprocessor_beans
    )

    evaluate_multi(
        retail_size_subsets,
        "Retail",
        best_params_retail,
        preprocessor_retail
    )

    results_size_multi_df = pd.DataFrame(results_multi)

    results_size_multi_df.to_csv(size_results_path, index=False)
    ran_now = True

    print("💾 Multiclass dataset size results saved")

# STEP 4 — Download if new

if ran_now:
    try:
        from google.colab import files
        print("⬇️ Downloading multiclass results...")
        files.download(size_results_path)
    except:
        print("Download skipped")
else:
    print("📁 No download needed (loaded from GitHub/local)")

🌐 Checking GitHub for multiclass dataset size results...
✅ Loaded from GitHub
   Source: https://raw.githubusercontent.com/Ilaha-Habibova/Tree-algorithms-dataset-characteristics/main/results/dataset_size_results_multiclass.csv
📁 No download needed (loaded from GitHub/local)


## **Dataset Size Experiment – Results visualization**

In [12]:
from IPython.display import display, HTML
import pandas as pd
import uuid

# 1. CONSOLIDATED STYLE AND SCRIPTS
display(HTML("""
<link rel="stylesheet" href="https://cdn.datatables.net/1.13.8/css/jquery.dataTables.min.css">
<script src="https://code.jquery.com/jquery-3.7.1.min.js"></script>
<script src="https://cdn.datatables.net/1.13.8/js/jquery.dataTables.min.js"></script>

<style>
.dataset-block { width: 100%; display: flex; flex-direction: column; align-items: flex-start; margin: 10px 0; }
.dataset-title { text-align: left !important; color: #0b3d91; font-weight: bold; font-size: 16px; margin-bottom: 4px; }
.size-title { text-align: left !important; font-weight: bold; font-size: 13px; margin-bottom: 2px; }

table.dataTable.custom-table {
    margin-left: 0 !important; margin-right: auto !important;
    border-collapse: collapse !important; font-size: 12px !important;
    width: auto !important; table-layout: auto !important; border: 1px solid #888 !important;
}

table.dataTable.custom-table thead th {
    position: relative;
    padding: 4px 28px 4px 10px !important;
    background-color: #dbeaf7 !important;
    text-align: center !important;
    border: 1px solid #bbb !important;
    color: #000000 !important;
    white-space: nowrap;
    background-image: none !important;
}

table.dataTable thead th.sorting:before,
table.dataTable thead th.sorting:after,
table.dataTable thead th.sorting_asc:before,
table.dataTable thead th.sorting_asc:after,
table.dataTable thead th.sorting_desc:before,
table.dataTable thead th.sorting_desc:after {
    display: none !important;
}

table.dataTable.custom-table thead th.sorting {
    background-image: url("data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' width='14' height='14' viewBox='0 0 24 24' fill='none' stroke='black' stroke-width='2'%3E%3Cpath d='M7 15l5 5 5-5M7 9l5-5 5 5'/%3E%3C/svg%3E") !important;
    background-repeat: no-repeat !important; background-position: center right 6px !important; background-size: 14px 14px !important;
}

table.dataTable.custom-table thead th.sorting_asc {
    background-image: url("data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' width='14' height='14' viewBox='0 0 24 24' fill='none' stroke='black' stroke-width='2.5'%3E%3Cpath d='M18 15l-6-6-6 6'/%3E%3C/svg%3E") !important;
}

table.dataTable.custom-table thead th.sorting_desc {
    background-image: url("data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' width='14' height='14' viewBox='0 0 24 24' fill='none' stroke='black' stroke-width='2.5'%3E%3Cpath d='M6 9l6 6 6-6'/%3E%3C/svg%3E") !important;
}

table.dataTable.custom-table tbody td {
    padding: 2px 8px !important; border: 1px solid #ddd !important;
    text-align: center !important; background-color: #f4f8fc !important;
    color: #000000 !important; line-height: 1.1 !important;
}
table.dataTable.custom-table tbody td:first-child { text-align: left !important; font-weight: bold; }

.dataTables_wrapper .dataTables_filter,
.dataTables_wrapper .dataTables_length,
.dataTables_wrapper .dataTables_info,
.dataTables_wrapper .dataTables_paginate { display: none !important; }
</style>
"""))

display(HTML("<h3 style='text-align: left; margin-bottom: 10px;'>DATASET SIZE RESULTS — MULTI-CLASS</h3>"))

for dataset in results_size_multi_df["dataset"].unique():

    dataset_df = results_size_multi_df[results_size_multi_df["dataset"] == dataset]

    display(HTML(f"<div class='dataset-block'>"))
    display(HTML(f"<div class='dataset-title'>{dataset.upper()}</div>"))

    for size in sorted(dataset_df["size"].unique()):

        subset = dataset_df[dataset_df["size"] == size]

        columns = [
            "model",
            "macro_f1",
            "macro_precision",
            "macro_recall",
            "balanced_accuracy",
            "training_time",
            "model_size_kb"
        ]

        table = subset[columns].round(4)\
            .sort_values(by="macro_f1", ascending=False)\
            .reset_index(drop=True)

        table_id = f"tbl_{uuid.uuid4().hex[:10]}"
        table_html = table.to_html(index=False, table_id=table_id, classes="display custom-table")

        html_block = f"""
            <div class="size-title">Dataset size = {size}</div>
            {table_html}
            <div style="height:12px;"></div>

            <script>
            (function() {{
                function initTable() {{
                    if (window.jQuery && $.fn.DataTable) {{
                        if (!$.fn.DataTable.isDataTable("#{table_id}")) {{
                            $("#{table_id}").DataTable({{
                                paging: false,
                                searching: false,
                                info: false,
                                order: [[1, "desc"]],
                                autoWidth: false,
                                columnDefs: [{{
                                    targets: "_all",
                                    orderSequence: ["desc", "asc", ""]
                                }}]
                            }});
                        }}
                    }} else {{
                        setTimeout(initTable, 100);
                    }}
                }}
                initTable();
            }})();
            </script>
        """

        display(HTML(html_block))

    display(HTML("</div>"))

model,macro_f1,macro_precision,macro_recall,balanced_accuracy,training_time,model_size_kb
LightGBM,0.9309,0.9364,0.9281,0.9281,0.5008,2510.1465
CatBoost,0.9279,0.9325,0.9264,0.9264,2.8560,655.5732
XGBoost,0.9273,0.9310,0.9265,0.9265,0.4155,961.2188
GradientBoosting,0.9214,0.9265,0.9189,0.9189,4.0731,4939.0615
ExtraTrees,0.9181,0.9257,0.9150,0.9150,0.2116,7223.7979
RandomForest,0.9072,0.9128,0.9049,0.9049,0.2719,1030.3350
DecisionTree,0.8877,0.8940,0.8857,0.8857,0.0118,12.0557


model,macro_f1,macro_precision,macro_recall,balanced_accuracy,training_time,model_size_kb
CatBoost,0.9352,0.9367,0.9344,0.9344,3.6545,656.1201
XGBoost,0.9332,0.9357,0.9314,0.9314,0.8523,1374.3936
GradientBoosting,0.9307,0.9338,0.9285,0.9285,9.0233,5313.4375
LightGBM,0.9287,0.9311,0.9269,0.9269,0.8587,3410.6875
ExtraTrees,0.9285,0.9313,0.9265,0.9265,0.3630,18914.1299
RandomForest,0.9262,0.9294,0.9241,0.9241,0.7171,2650.5371
DecisionTree,0.9112,0.9119,0.9112,0.9112,0.0420,25.1807


model,macro_f1,macro_precision,macro_recall,balanced_accuracy,training_time,model_size_kb
CatBoost,0.9388,0.9405,0.9375,0.9375,5.3760,656.3389
XGBoost,0.9384,0.9406,0.9365,0.9365,1.2725,1869.6348
LightGBM,0.9376,0.9395,0.9359,0.9359,1.7254,4521.0986
GradientBoosting,0.9350,0.9373,0.9332,0.9332,24.5470,5408.3936
RandomForest,0.9333,0.9354,0.9316,0.9316,2.6138,6727.8418
ExtraTrees,0.9309,0.9332,0.9291,0.9291,0.8116,45843.9346
DecisionTree,0.9159,0.9182,0.9141,0.9141,0.1386,52.1367


model,macro_f1,macro_precision,macro_recall,balanced_accuracy,training_time,model_size_kb
CatBoost,0.7499,0.7632,0.7404,0.7404,0.9730,429.6201
RandomForest,0.7478,0.7676,0.7347,0.7347,0.5887,5652.0791
XGBoost,0.7384,0.7496,0.7307,0.7307,0.3146,1295.0605
GradientBoosting,0.7356,0.7544,0.7214,0.7214,2.1289,3264.8926
LightGBM,0.7290,0.7387,0.7240,0.7240,0.4613,2705.0625
ExtraTrees,0.6702,0.7070,0.6506,0.6506,0.2465,13164.6152
DecisionTree,0.6558,0.6588,0.6589,0.6589,0.0156,21.6650


model,macro_f1,macro_precision,macro_recall,balanced_accuracy,training_time,model_size_kb
CatBoost,0.7399,0.7526,0.7306,0.7306,1.3857,429.7607
XGBoost,0.7374,0.7492,0.7281,0.7281,0.4941,1334.7178
LightGBM,0.7343,0.7496,0.7225,0.7225,0.5105,2680.0918
RandomForest,0.7309,0.7496,0.7172,0.7172,1.5188,16365.1377
GradientBoosting,0.7308,0.7455,0.7194,0.7194,4.1073,3294.2891
ExtraTrees,0.6967,0.7327,0.6777,0.6777,0.4874,36927.1523
DecisionTree,0.6792,0.6815,0.6792,0.6792,0.0268,49.6055


model,macro_f1,macro_precision,macro_recall,balanced_accuracy,training_time,model_size_kb
CatBoost,0.7649,0.7769,0.7550,0.7550,2.3834,429.5029
XGBoost,0.7622,0.7743,0.7521,0.7521,0.7223,1345.7598
GradientBoosting,0.7621,0.7745,0.7516,0.7516,10.4709,3332.2666
LightGBM,0.7602,0.7718,0.7504,0.7504,0.7821,2678.3057
RandomForest,0.7532,0.7681,0.7414,0.7414,4.7117,45868.7666
ExtraTrees,0.7245,0.7486,0.7085,0.7085,1.1949,92908.5674
DecisionTree,0.7090,0.7135,0.7053,0.7053,0.0812,92.1768


In [13]:
# VISUALIZATION — SIZE-SPECIFIC RANKINGS
from IPython.display import display, HTML

MODEL_DISPLAY_NAMES = {
    "DecisionTree": "Decision Tree",
    "RandomForest": "Random Forest",
    "ExtraTrees": "Extra Trees",
    "GradientBoosting": "Gradient Boosting",
    "XGBoost": "XGBoost",
    "LightGBM": "LightGBM",
    "CatBoost": "CatBoost"
}

def pretty_model_name(name):
    return MODEL_DISPLAY_NAMES.get(name, name)

# 1. Styles
style_html = """
<style>
    .condition-box {
        font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
        margin: 12px 0;
        padding: 10px 12px;
        background: #ffffff;
        border: 1px solid #d9e3f0;
        border-radius: 8px;
        width: 340px;
        box-shadow: 1px 1px 4px rgba(0,0,0,0.04);
    }
    .condition-title {
        font-size: 14px;
        font-weight: 700;
        color: #0b3d91;
        border-bottom: 2px solid #7ea6d8;
        margin-bottom: 8px;
        padding-bottom: 4px;
        text-transform: uppercase;
        letter-spacing: 0.2px;
    }
    .rank-table {
        width: 100%;
        border-collapse: collapse;
    }
    .rank-row {
        border-bottom: 1px solid #edf2f7;
    }
    .rank-row:last-child {
        border-bottom: none;
    }
    .rank-cell {
        padding: 5px 3px;
        font-size: 13px;
        color: #000 !important;
        line-height: 1.15;
    }
    .rank-icon {
        width: 34px;
        text-align: center;
        font-weight: 700;
    }
    .rank-model {
        font-weight: 600;
    }
    .rank-val {
        text-align: right;
        font-family: Consolas, 'Courier New', monospace;
        font-weight: 700;
        width: 56px;
    }
</style>
"""
display(HTML(style_html))
display(HTML("<h2 style='color:#000; margin: 6px 0 10px 0; font-size: 22px;'>📊 Size-specific rankings — multi-class (Macro-F1)</h2>"))

# 2. Logic and Rendering
sizes = sorted(results_size_multi_df["size"].unique())

for size in sizes:
    df = results_size_multi_df[
        results_size_multi_df["size"] == size
    ].copy()

    df["rank"] = df.groupby("dataset")["macro_f1"].rank(
        ascending=False,
        method="min"
    )

    avg_rank = df.groupby("model")["rank"].mean().sort_values().round(4)

    rank_df = avg_rank.reset_index()
    rank_df.columns = ["model", "avg_rank"]

    rank_df["final_rank"] = rank_df["avg_rank"].rank(
        method="dense"
    ).astype(int)

    html_output = f"""
    <div class="condition-box">
        <div class="condition-title">Dataset size = {size}</div>
        <table class="rank-table">
    """

    for _, row in rank_df.iterrows():
        r = int(row["final_rank"])
        icon = {1: "🥇", 2: "🥈", 3: "🥉"}.get(r, f"#{r}")
        model_display = pretty_model_name(row["model"])

        html_output += f"""
        <tr class="rank-row">
            <td class="rank-cell rank-icon">{icon}</td>
            <td class="rank-cell rank-model">{model_display}</td>
            <td class="rank-cell rank-val">{row['avg_rank']:.2f}</td>
        </tr>
        """

    html_output += "</table></div>"
    display(HTML(html_output))

🥇,CatBoost,1.50
🥈,LightGBM,3.00
🥈,XGBoost,3.00
🥉,Gradient Boosting,4.00
🥉,Random Forest,4.00
#4,Extra Trees,5.50
#5,Decision Tree,7.00


🥇,CatBoost,1.00
🥈,XGBoost,2.00
🥉,LightGBM,3.50
#4,Gradient Boosting,4.00
#5,Random Forest,5.00
#6,Extra Trees,5.50
#7,Decision Tree,7.00


🥇,CatBoost,1.00
🥈,XGBoost,2.00
🥉,LightGBM,3.50
🥉,Gradient Boosting,3.50
#4,Random Forest,5.00
#5,Extra Trees,6.00
#6,Decision Tree,7.00


In [14]:
from IPython.display import display, HTML
import pandas as pd

MODEL_DISPLAY_NAMES = {
    "DecisionTree": "Decision Tree",
    "RandomForest": "Random Forest",
    "ExtraTrees": "Extra Trees",
    "GradientBoosting": "Gradient Boosting",
    "XGBoost": "XGBoost",
    "LightGBM": "LightGBM",
    "CatBoost": "CatBoost"
}

def pretty_model_name(name):
    return MODEL_DISPLAY_NAMES.get(name, name)

# STYLE
style_html = """
<style>
    .ranking-container {
        font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
        margin-top: 12px;
        padding: 10px 12px;
        background-color: #ffffff;
        border-radius: 8px;
        border: 1px solid #d9e3f0;
        width: fit-content;
        min-width: 280px;
        box-shadow: 1px 1px 4px rgba(0,0,0,0.04);
    }
    .ranking-header {
        color: #0b3d91;
        font-size: 14px;
        font-weight: 700;
        border-bottom: 2px solid #7ea6d8;
        margin-bottom: 8px;
        padding-bottom: 4px;
        text-transform: uppercase;
        letter-spacing: 0.2px;
    }
    .ranking-table {
        width: 100%;
        border-collapse: collapse;
    }
    .ranking-row {
        border-bottom: 1px solid #edf2f7;
    }
    .ranking-row:last-child {
        border-bottom: none;
    }
    .rank-num {
        font-weight: 700;
        font-size: 13px;
        width: 34px;
        text-align: center;
        color: #000000 !important;
        padding: 6px 2px;
        line-height: 1.1;
    }
    .model-name {
        padding: 6px 6px;
        font-size: 13px;
        font-weight: 600;
        color: #000000 !important;
        line-height: 1.1;
    }
    .avg-rank-val {
        text-align: right;
        padding: 6px 4px 6px 10px;
        font-family: Consolas, 'Courier New', monospace;
        font-size: 13px;
        font-weight: 700;
        color: #000000 !important;
        min-width: 52px;
        line-height: 1.1;
    }
</style>
"""

display(HTML(style_html))
display(HTML("<h2 style='color:#000; margin: 6px 0 10px 5px; font-size: 22px;'>🏆 Global ranking — multi-class (all datasets)</h2>"))

df = results_size_multi_df.copy()

df["rank"] = (
    df.groupby(["dataset", "size"])["macro_f1"]
    .rank(ascending=False, method="min")
)

global_rank = (
    df.groupby("model")["rank"]
    .mean()
    .sort_values()
    .round(4)
)

rank_df = global_rank.reset_index()
rank_df.columns = ["model", "avg_rank"]

rank_df["final_rank"] = (
    rank_df["avg_rank"]
    .rank(method="dense")
    .astype(int)
)

html = """
<div class="ranking-container">
    <div class="ranking-header">
        Global performance (Macro-F1)
    </div>
    <table class="ranking-table">
"""

for _, row in rank_df.iterrows():
    r = int(row["final_rank"])
    rank_display = {
        1: "🥇",
        2: "🥈",
        3: "🥉"
    }.get(r, f"#{r}")

    model_display = pretty_model_name(row["model"])

    html += f"""
    <tr class="ranking-row">
        <td class="rank-num">{rank_display}</td>
        <td class="model-name">{model_display}</td>
        <td class="avg-rank-val">{row['avg_rank']:.2f}</td>
    </tr>
    """

html += """
    </table>
</div>
"""

display(HTML(html))

🥇,CatBoost,1.17
🥈,XGBoost,2.33
🥉,LightGBM,3.33
#4,Gradient Boosting,3.83
#5,Random Forest,4.67
#6,Extra Trees,5.67
#7,Decision Tree,7.00


## 11. Class Imbalance Manipulation

This section investigates how classification algorithms behave when class distributions become increasingly imbalanced.

The severity of imbalance is measured using the **Imbalance Ratio (IR)**:

\[
IR = \frac{\text{largest class size}}{\text{smallest class size}}
\]

Three imbalance levels are examined:

- IR = 1 — balanced baseline condition  
- IR = 4 — moderate imbalance  
- IR = 9 — severe imbalance  

To ensure that performance differences arise only from changes in class distribution, the dataset size is kept constant in all imbalance experiments:

\[
N = 3600
\]

For multi-class datasets, imbalance is generated while preserving the original class-size ordering. This means that the largest class remains the largest, the smallest class remains the smallest, and intermediate classes preserve their relative positions.

Each imbalance condition represents a separate dataset version that is later used in model evaluation.

In [15]:
# PART 1 — Class Imbalance Setup (MULTI-CLASS)
import os
import numpy as np
import pandas as pd

N_IMBALANCE = 3600
imbalance_levels = [1, 4, 9]

IMBALANCE_DATASETS_PATH = os.path.join(
    BASE_PATH,
    "datasets",
    "generated_datasets",
    "imbalance_experiments"
)

os.makedirs(IMBALANCE_DATASETS_PATH, exist_ok=True)

# File paths

beans_files = {
    ir: os.path.join(IMBALANCE_DATASETS_PATH, f"beans_IR_{ir}.csv")
    for ir in imbalance_levels
}

retail_files = {
    ir: os.path.join(IMBALANCE_DATASETS_PATH, f"retail_IR_{ir}.csv")
    for ir in imbalance_levels
}

# Detect distributions

def get_class_info(y):
    vc = pd.Series(y).value_counts().sort_values(ascending=False)
    return vc.index.tolist(), vc.values.tolist()

bean_labels, bean_counts = get_class_info(y_beans)
retail_labels, retail_counts = get_class_info(y_retail)

# Sampling
def sample_by_class_counts(X, y, class_counts, seed=SEED):

    X = pd.DataFrame(X)
    y = pd.Series(y)

    rng = np.random.RandomState(seed)
    sampled = []

    for cls, count in class_counts.items():
        idx = y[y == cls].index
        chosen = rng.choice(idx, size=count, replace=False)
        sampled.extend(chosen)

    sampled = np.array(sampled)

    return (
        X.loc[sampled].reset_index(drop=True),
        y.loc[sampled].reset_index(drop=True)
    )

# Multi-class imbalance logic

def linear_rank_preserving_counts(labels, counts, N, R):

    counts = np.array(counts, dtype=float)

    t = (counts - counts.min()) / (counts.max() - counts.min())
    base = 1 + (R - 1) * t

    raw = (N / base.sum()) * base
    final = np.floor(raw).astype(int)

    remainder = N - final.sum()
    order = np.argsort(-(raw - final))

    for i in range(remainder):
        final[order[i]] += 1

    return dict(zip(labels, final))

In [16]:

# CLASS IMBALANCE — MULTI-CLASS

import requests, zipfile

ZIP_URL = "https://raw.githubusercontent.com/Ilaha-Habibova/Tree-algorithms-dataset-characteristics/main/datasets/generated_datasets/imbalance_experiments/multiclass_imbalance_subsets.zip"

zip_path = os.path.join(IMBALANCE_DATASETS_PATH, "multiclass_imbalance_subsets.zip")

# STEP 1 — GitHub

print("🌐 Checking GitHub...")

github_ok = False

try:
    r = requests.get(ZIP_URL)

    if r.status_code == 200:
        with open(zip_path, "wb") as f:
            f.write(r.content)

        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(IMBALANCE_DATASETS_PATH)

        github_ok = True

        print("🌐 Loaded from GitHub")
        print(f"   Source: {ZIP_URL}")

    else:
        print("❌ GitHub not available")

except:
    print("⚠️ GitHub error")

# STEP 2 — CHECK
all_files = list(beans_files.values()) + list(retail_files.values())
files_exist = all(os.path.exists(f) for f in all_files)

# STEP 3 — LOAD

if files_exist:

    source = "GitHub" if github_ok else "Local"
    print(f"📂 Loading ({source})")

    beans_imbalance_subsets = {}
    retail_imbalance_subsets = {}

    for ir in imbalance_levels:

        df = pd.read_csv(beans_files[ir])
        beans_imbalance_subsets[ir] = (
            df.drop(columns=["Class"]),
            df["Class"]
        )

        df = pd.read_csv(retail_files[ir])
        retail_imbalance_subsets[ir] = (
            df.drop(columns=["customer_segment"]),
            df["customer_segment"]
        )

    print("📁 No download needed")

# STEP 4 — GENERATE
else:

    print("🚀 Generating multiclass imbalance datasets...")

    beans_imbalance_subsets = {}
    retail_imbalance_subsets = {}

    for ir in imbalance_levels:

        counts = linear_rank_preserving_counts(bean_labels, bean_counts, N_IMBALANCE, ir)
        Xb, yb = sample_by_class_counts(X_beans, y_beans, counts)
        beans_imbalance_subsets[ir] = (Xb, yb)

        pd.concat([Xb, yb], axis=1).to_csv(beans_files[ir], index=False)

        counts = linear_rank_preserving_counts(retail_labels, retail_counts, N_IMBALANCE, ir)
        Xr, yr = sample_by_class_counts(X_retail, y_retail, counts)
        retail_imbalance_subsets[ir] = (Xr, yr)

        pd.concat([Xr, yr], axis=1).to_csv(retail_files[ir], index=False)

    zip_file = os.path.join(BASE_PATH, "multiclass_imbalance_subsets.zip")

    with zipfile.ZipFile(zip_file, 'w') as z:
        for ir in imbalance_levels:
            z.write(beans_files[ir], os.path.basename(beans_files[ir]))
            z.write(retail_files[ir], os.path.basename(retail_files[ir]))

    print("📦 Clean ZIP created")

    try:
        from google.colab import files
        files.download(zip_file)
    except:
        pass

🌐 Checking GitHub...
🌐 Loaded from GitHub
   Source: https://raw.githubusercontent.com/Ilaha-Habibova/Tree-algorithms-dataset-characteristics/main/datasets/generated_datasets/imbalance_experiments/multiclass_imbalance_subsets.zip
📂 Loading (GitHub)
📁 No download needed


## 12. Class Imbalance Experiment – Model Evaluation

This section evaluates the performance of the seven machine learning algorithms under different class imbalance conditions.

Each imbalance level is treated as a separate dataset version.

The same evaluation procedure used in the dataset size experiment is applied here:

- stratified five-fold cross-validation  
- Macro F1-score as the primary evaluation metric  
- macro precision, macro recall, and balanced accuracy as complementary metrics  
- measurement of training time and model size

The tuned hyperparameters obtained earlier are reused so that algorithm comparisons remain fair and consistent across all generated dataset versions.

In [17]:

# CLASS IMBALANCE RESULTS — MULTI-CLASS

import os
import requests
import pandas as pd

RESULTS_PATH = os.path.join(BASE_PATH, "results")
os.makedirs(RESULTS_PATH, exist_ok=True)

IMBALANCE_RESULTS_URL = "https://raw.githubusercontent.com/Ilaha-Habibova/Tree-algorithms-dataset-characteristics/main/results/imbalance_results_multiclass.csv"

imbalance_results_path = os.path.join(
    RESULTS_PATH,
    "imbalance_results_multiclass.csv"
)

print("🌐 Checking GitHub for multiclass imbalance results...")

results_imbalance_multi_df = None
ran_now = False

# STEP 1 — GitHub

try:
    r = requests.get(IMBALANCE_RESULTS_URL)

    if r.status_code == 200:
        with open(imbalance_results_path, "wb") as f:
            f.write(r.content)

        results_imbalance_multi_df = pd.read_csv(imbalance_results_path)

        print(f"✅ Loaded from GitHub: {IMBALANCE_RESULTS_URL}")

    else:
        print("❌ GitHub not available")

except:
    print("⚠️ GitHub error")

# STEP 2 — Local
if results_imbalance_multi_df is None and os.path.exists(imbalance_results_path):
    print("📂 Loading multiclass results from local...")
    results_imbalance_multi_df = pd.read_csv(imbalance_results_path)

# STEP 3 — Run if needed

if results_imbalance_multi_df is None:

    print("🚀 Running multiclass imbalance experiment...")

    results_multi = []

    def evaluate_multi(subsets, dataset_name, best_params, preprocessor):

        for IR in sorted(subsets.keys()):

            X_subset, y_subset = subsets[IR]

            for model_name, base_model in models.items():

                model = clone(base_model)

                if model_name in best_params:
                    tuned_params = {
                        k.replace("model__", ""): v
                        for k, v in best_params[model_name]["best_params"].items()
                    }
                    model.set_params(**tuned_params)

                pipeline = Pipeline([
                    ("preprocessing", preprocessor),
                    ("model", model)
                ])

                scoring = {
                    "balanced_accuracy": "balanced_accuracy",
                    "macro_f1": "f1_macro",
                    "macro_precision": "precision_macro",
                    "macro_recall": "recall_macro"
                }

                cv_results = cross_validate(
                    pipeline,
                    X_subset,
                    y_subset,
                    cv=cv,
                    scoring=scoring,
                    n_jobs=1
                )

                pipeline.fit(X_subset, y_subset)

                results_multi.append({
                    "dataset": dataset_name,
                    "imbalance_ratio": IR,
                    "model": model_name,
                    "balanced_accuracy": cv_results["test_balanced_accuracy"].mean(),
                    "macro_f1": cv_results["test_macro_f1"].mean(),
                    "macro_precision": cv_results["test_macro_precision"].mean(),
                    "macro_recall": cv_results["test_macro_recall"].mean(),
                    "training_time": cv_results["fit_time"].mean(),
                    "model_size_kb": get_model_size_kb(pipeline.named_steps["model"])
                })

    evaluate_multi(
        beans_imbalance_subsets,
        "DryBean",
        best_params_beans,
        preprocessor_beans
    )

    evaluate_multi(
        retail_imbalance_subsets,
        "Retail",
        best_params_retail,
        preprocessor_retail
    )

    results_imbalance_multi_df = pd.DataFrame(results_multi)

    results_imbalance_multi_df.to_csv(imbalance_results_path, index=False)
    ran_now = True

    print("💾 Multiclass imbalance results saved")

# STEP 4 — Download if new

if ran_now:
    try:
        from google.colab import files
        print("⬇️ Downloading multiclass results...")
        files.download(imbalance_results_path)
    except:
        print("Download skipped")
else:
    print("📁 No download needed (loaded from GitHub/local)")

🌐 Checking GitHub for multiclass imbalance results...
✅ Loaded from GitHub: https://raw.githubusercontent.com/Ilaha-Habibova/Tree-algorithms-dataset-characteristics/main/results/imbalance_results_multiclass.csv
📁 No download needed (loaded from GitHub/local)


## **Imbalance Experiment - Visualization**

In [18]:
from IPython.display import display, HTML
import pandas as pd
import uuid

# CSS & STYLING
display(HTML("""
<link rel="stylesheet" href="https://cdn.datatables.net/1.13.8/css/jquery.dataTables.min.css">
<script src="https://code.jquery.com/jquery-3.7.1.min.js"></script>
<script src="https://cdn.datatables.net/1.13.8/js/jquery.dataTables.min.js"></script>

<style>
.dataset-block { width: 100%; display: flex; flex-direction: column; align-items: flex-start; margin: 10px 0; }
.dataset-title { text-align: left !important; color: #0b3d91; font-weight: bold; font-size: 16px; margin-bottom: 4px; }
.size-title { text-align: left !important; font-weight: bold; font-size: 13px; margin-bottom: 2px; }

table.dataTable.custom-table {
    margin-left: 0 !important; margin-right: auto !important;
    border-collapse: collapse !important; font-size: 12px !important;
    width: auto !important; table-layout: auto !important;
    border: 1px solid #888 !important;
}

table.dataTable.custom-table thead th,
table.dataTable.custom-table thead th.sorting,
table.dataTable.custom-table thead th.sorting_asc,
table.dataTable.custom-table thead th.sorting_desc {
    position: relative;
    padding: 4px 28px 4px 10px !important;
    background-color: #dbeaf7 !important;
    text-align: center !important;
    border: 1px solid #bbb !important;
    background-image: none !important;
    color: #000000 !important;
}

table.dataTable thead th.sorting:before,
table.dataTable thead th.sorting:after,
table.dataTable thead th.sorting_asc:before,
table.dataTable thead th.sorting_asc:after,
table.dataTable thead th.sorting_desc:before,
table.dataTable thead th.sorting_desc:after {
    display: none !important;
}

table.dataTable.custom-table tbody td {
    padding: 2px 8px !important;
    border: 1px solid #ddd !important;
    text-align: center !important;
    background-color: #f4f8fc !important;
    color: #000000 !important;
}

table.dataTable.custom-table tbody td:first-child {
    text-align: left !important;
    font-weight: bold;
}

table.dataTable.custom-table thead th.sorting {
    background-image: url("data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' width='14' height='14' viewBox='0 0 24 24' fill='none' stroke='black' stroke-width='2'%3E%3Cpath d='M7 15l5 5 5-5M7 9l5-5 5 5'/%3E%3C/svg%3E") !important;
    background-repeat: no-repeat !important;
    background-position: center right 6px !important;
}

table.dataTable.custom-table thead th.sorting_asc {
    background-image: url("data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' width='14' height='14' viewBox='0 0 24 24' fill='none' stroke='black' stroke-width='2.5'%3E%3Cpath d='M18 15l-6-6-6 6'/%3E%3C/svg%3E") !important;
}

table.dataTable.custom-table thead th.sorting_desc {
    background-image: url("data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' width='14' height='14' viewBox='0 0 24 24' fill='none' stroke='black' stroke-width='2.5'%3E%3Cpath d='M6 9l6 6 6-6'/%3E%3C/svg%3E") !important;
}

.dataTables_wrapper .dataTables_filter,
.dataTables_wrapper .dataTables_length,
.dataTables_wrapper .dataTables_info,
.dataTables_wrapper .dataTables_paginate {
    display: none !important;
}
</style>
"""))

display(HTML("<h3 style='text-align: left;'>CLASS IMBALANCE RESULTS — MULTI-CLASS</h3>"))

for dataset in results_imbalance_multi_df["dataset"].unique():

    dataset_df = results_imbalance_multi_df[
        results_imbalance_multi_df["dataset"] == dataset
    ]

    display(HTML("<div class='dataset-block'>"))
    display(HTML(f"<div class='dataset-title'>{dataset.upper()}</div>"))

    for IR in sorted(dataset_df["imbalance_ratio"].unique()):

        subset = dataset_df[
            dataset_df["imbalance_ratio"] == IR
        ]

        columns = [
            "model",
            "macro_f1",
            "macro_precision",
            "macro_recall",
            "balanced_accuracy",
            "training_time",
            "model_size_kb"
        ]

        table = subset[columns].round(4)\
            .sort_values(by="macro_f1", ascending=False)\
            .reset_index(drop=True)

        table_id = f"tbl_{uuid.uuid4().hex[:10]}"
        table_html = table.to_html(index=False, table_id=table_id, classes="display custom-table")

        html_block = f"""
        <div class="size-title">Imbalance Ratio = {IR}</div>
        {table_html}
        <div style="height:12px;"></div>

        <script>
        (function() {{
            function initTable() {{
                if (window.jQuery && $.fn.DataTable) {{
                    if (!$.fn.DataTable.isDataTable("#{table_id}")) {{
                        $("#{table_id}").DataTable({{
                            paging: false,
                            searching: false,
                            info: false,
                            order: [[1, "desc"]],
                            autoWidth: false,
                            columnDefs: [{{
                                targets: "_all",
                                orderSequence: ["desc", "asc", ""]
                            }}]
                        }});
                    }}
                }} else {{
                    setTimeout(initTable, 100);
                }}
            }}
            initTable();
        }})();
        </script>
        """

        display(HTML(html_block))

    display(HTML("</div>"))

model,macro_f1,macro_precision,macro_recall,balanced_accuracy,training_time,model_size_kb
CatBoost,0.9381,0.9386,0.9381,0.9381,6.2055,656.0264
GradientBoosting,0.9350,0.9356,0.9350,0.9350,16.4777,5257.1719
XGBoost,0.9326,0.9331,0.9325,0.9325,1.3701,1398.3135
LightGBM,0.9325,0.9329,0.9326,0.9326,1.4863,3355.4082
ExtraTrees,0.9297,0.9300,0.9298,0.9298,0.5710,19770.7705
RandomForest,0.9281,0.9287,0.9281,0.9281,1.4930,2718.9863
DecisionTree,0.9074,0.9078,0.9075,0.9075,0.1987,26.1182


model,macro_f1,macro_precision,macro_recall,balanced_accuracy,training_time,model_size_kb
CatBoost,0.9435,0.9450,0.9426,0.9426,5.9699,656.0732
GradientBoosting,0.9332,0.9361,0.9313,0.9313,16.5040,5234.7422
XGBoost,0.9329,0.9348,0.9315,0.9315,1.4561,1425.7979
LightGBM,0.9328,0.9342,0.9321,0.9321,1.4659,3603.3398
ExtraTrees,0.9285,0.9307,0.9269,0.9269,0.5844,20950.1455
RandomForest,0.9283,0.9303,0.9271,0.9271,1.5050,2953.6953
DecisionTree,0.9021,0.9051,0.9020,0.9020,0.0681,29.8682


model,macro_f1,macro_precision,macro_recall,balanced_accuracy,training_time,model_size_kb
CatBoost,0.9389,0.9397,0.9389,0.9389,5.9259,656.0342
LightGBM,0.9313,0.9330,0.9304,0.9304,1.6704,3626.7979
GradientBoosting,0.9302,0.9326,0.9284,0.9284,16.6258,5281.5381
XGBoost,0.9295,0.9314,0.9283,0.9283,1.4776,1427.9189
RandomForest,0.9280,0.9305,0.9266,0.9266,1.4988,3010.4404
ExtraTrees,0.9243,0.9277,0.9221,0.9221,0.5856,21660.3018
DecisionTree,0.9025,0.9058,0.9009,0.9009,0.0706,28.6963


model,macro_f1,macro_precision,macro_recall,balanced_accuracy,training_time,model_size_kb
CatBoost,0.7491,0.7545,0.7472,0.7472,2.3553,429.5654
RandomForest,0.7365,0.7438,0.7342,0.7342,2.8877,18785.4023
GradientBoosting,0.7358,0.7416,0.7333,0.7333,7.2550,3325.0625
XGBoost,0.7349,0.7404,0.7325,0.7325,0.7193,1330.3535
LightGBM,0.7282,0.7331,0.7258,0.7258,1.0429,2675.9307
ExtraTrees,0.7167,0.7179,0.7167,0.7167,0.8751,42112.1426
DecisionTree,0.6766,0.6798,0.6750,0.6750,0.0660,54.6680


model,macro_f1,macro_precision,macro_recall,balanced_accuracy,training_time,model_size_kb
XGBoost,0.7401,0.7565,0.7271,0.7271,0.7903,1334.8945
GradientBoosting,0.7398,0.7580,0.7257,0.7257,7.1595,3276.6758
CatBoost,0.7392,0.7543,0.7276,0.7276,2.4014,429.5811
RandomForest,0.7386,0.7607,0.7223,0.7223,2.8530,19378.7461
LightGBM,0.7343,0.7520,0.7202,0.7202,0.9231,2677.8496
ExtraTrees,0.6913,0.7185,0.6759,0.6759,1.0289,43246.1396
DecisionTree,0.6688,0.6725,0.6666,0.6666,0.0456,59.5430


model,macro_f1,macro_precision,macro_recall,balanced_accuracy,training_time,model_size_kb
CatBoost,0.7303,0.7636,0.7060,0.7060,2.3857,429.7686
LightGBM,0.7209,0.7507,0.6990,0.6990,0.9803,2680.4697
GradientBoosting,0.7193,0.7569,0.6937,0.6937,7.3051,3242.5918
RandomForest,0.7186,0.7657,0.6883,0.6883,2.8450,19212.3281
XGBoost,0.7186,0.7460,0.6979,0.6979,0.9061,1321.0840
ExtraTrees,0.6652,0.7445,0.6305,0.6305,0.9179,42260.6045
DecisionTree,0.6646,0.6762,0.6562,0.6562,0.0450,55.0430


In [19]:
# VISUALIZATION — IMBALANCE-SPECIFIC RANKINGS
from IPython.display import display, HTML

MODEL_DISPLAY_NAMES = {
    "DecisionTree": "Decision Tree",
    "RandomForest": "Random Forest",
    "ExtraTrees": "Extra Trees",
    "GradientBoosting": "Gradient Boosting",
    "XGBoost": "XGBoost",
    "LightGBM": "LightGBM",
    "CatBoost": "CatBoost"
}

def pretty_model_name(name):
    return MODEL_DISPLAY_NAMES.get(name, name)

# 1. Styles
style_html = """
<style>
    .condition-box {
        font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
        margin: 12px 0;
        padding: 10px 12px;
        background: #ffffff;
        border: 1px solid #d9e3f0;
        border-radius: 8px;
        width: 340px;
        box-shadow: 1px 1px 4px rgba(0,0,0,0.04);
    }
    .condition-title {
        font-size: 14px;
        font-weight: 700;
        color: #0b3d91;
        border-bottom: 2px solid #7ea6d8;
        margin-bottom: 8px;
        padding-bottom: 4px;
        text-transform: uppercase;
        letter-spacing: 0.2px;
    }
    .rank-table {
        width: 100%;
        border-collapse: collapse;
    }
    .rank-row {
        border-bottom: 1px solid #edf2f7;
    }
    .rank-row:last-child {
        border-bottom: none;
    }
    .rank-cell {
        padding: 5px 3px;
        font-size: 13px;
        color: #000 !important;
        line-height: 1.15;
    }
    .rank-icon {
        width: 34px;
        text-align: center;
        font-weight: 700;
    }
    .rank-model {
        font-weight: 600;
    }
    .rank-val {
        text-align: right;
        font-family: Consolas, 'Courier New', monospace;
        font-weight: 700;
        width: 56px;
    }
</style>
"""
display(HTML(style_html))
display(HTML("<h2 style='color:#000; margin: 6px 0 10px 0; font-size: 22px;'>📊 Imbalance-specific rankings — multi-class (Macro-F1)</h2>"))

# 2. Logic and Rendering
for ir in sorted(results_imbalance_multi_df["imbalance_ratio"].unique()):

    df = results_imbalance_multi_df[
        results_imbalance_multi_df["imbalance_ratio"] == ir
    ].copy()

    df["rank"] = df.groupby("dataset")["macro_f1"].rank(
        ascending=False,
        method="min"
    )

    avg_rank = df.groupby("model")["rank"].mean().sort_values().round(4)

    rank_df = avg_rank.reset_index()
    rank_df.columns = ["model", "avg_rank"]
    rank_df["final_rank"] = rank_df["avg_rank"].rank(method="dense").astype(int)

    html_output = f"""
    <div class="condition-box">
        <div class="condition-title">Imbalance ratio = {ir}</div>
        <table class="rank-table">
    """

    for _, row in rank_df.iterrows():
        r = int(row["final_rank"])
        icon = {1: "🥇", 2: "🥈", 3: "🥉"}.get(r, f"#{r}")
        model_display = pretty_model_name(row["model"])

        html_output += f"""
        <tr class="rank-row">
            <td class="rank-cell rank-icon">{icon}</td>
            <td class="rank-cell rank-model">{model_display}</td>
            <td class="rank-cell rank-val">{row['avg_rank']:.2f}</td>
        </tr>
        """

    html_output += "</table></div>"
    display(HTML(html_output))

🥇,CatBoost,1.00
🥈,Gradient Boosting,2.50
🥉,XGBoost,3.50
#4,Random Forest,4.00
#5,LightGBM,4.50
#6,Extra Trees,5.50
#7,Decision Tree,7.00


🥇,CatBoost,2.00
🥇,Gradient Boosting,2.00
🥇,XGBoost,2.00
🥈,LightGBM,4.50
🥉,Random Forest,5.00
#4,Extra Trees,5.50
#5,Decision Tree,7.00


🥇,CatBoost,1.00
🥈,LightGBM,2.00
🥉,Gradient Boosting,3.00
#4,XGBoost,4.00
#5,Random Forest,5.00
#6,Extra Trees,6.00
#7,Decision Tree,7.00


In [20]:
from IPython.display import display, HTML
import pandas as pd

MODEL_DISPLAY_NAMES = {
    "DecisionTree": "Decision Tree",
    "RandomForest": "Random Forest",
    "ExtraTrees": "Extra Trees",
    "GradientBoosting": "Gradient Boosting",
    "XGBoost": "XGBoost",
    "LightGBM": "LightGBM",
    "CatBoost": "CatBoost"
}

def pretty_model_name(name):
    return MODEL_DISPLAY_NAMES.get(name, name)

# STYLE
style_html = """
<style>
    .ranking-container {
        font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
        margin-top: 12px;
        padding: 10px 12px;
        background-color: #ffffff;
        border-radius: 8px;
        border: 1px solid #d9e3f0;
        width: fit-content;
        min-width: 280px;
        box-shadow: 1px 1px 4px rgba(0,0,0,0.04);
    }
    .ranking-header {
        color: #0b3d91;
        font-size: 14px;
        font-weight: 700;
        border-bottom: 2px solid #7ea6d8;
        margin-bottom: 8px;
        padding-bottom: 4px;
        text-transform: uppercase;
        letter-spacing: 0.2px;
    }
    .ranking-table {
        width: 100%;
        border-collapse: collapse;
    }
    .ranking-row {
        border-bottom: 1px solid #edf2f7;
    }
    .ranking-row:last-child {
        border-bottom: none;
    }
    .rank-num {
        font-weight: 700;
        font-size: 13px;
        width: 34px;
        text-align: center;
        color: #000000 !important;
        padding: 6px 2px;
        line-height: 1.1;
    }
    .model-name {
        padding: 6px 6px;
        font-size: 13px;
        font-weight: 600;
        color: #000000 !important;
        line-height: 1.1;
    }
    .avg-rank-val {
        text-align: right;
        padding: 6px 4px 6px 10px;
        font-family: Consolas, 'Courier New', monospace;
        font-size: 13px;
        font-weight: 700;
        color: #000000 !important;
        min-width: 52px;
        line-height: 1.1;
    }
</style>
"""

display(HTML(style_html))
display(HTML("<h2 style='color:#000; margin: 6px 0 10px 5px; font-size: 22px;'>🏆 Global ranking — multi-class (all datasets)</h2>"))

# GLOBAL RANKING LOGIC (MULTI-CLASS)

df = results_imbalance_multi_df.copy()

df["rank"] = (
    df.groupby(["dataset", "imbalance_ratio"])["macro_f1"]
    .rank(ascending=False, method="min")
)

global_rank = (
    df.groupby("model")["rank"]
    .mean()
    .sort_values()
    .round(4)
)

rank_df = global_rank.reset_index()
rank_df.columns = ["model", "avg_rank"]

rank_df["final_rank"] = (
    rank_df["avg_rank"]
    .rank(method="dense")
    .astype(int)
)

html = """
<div class="ranking-container">
    <div class="ranking-header">
        Global performance (Macro-F1)
    </div>
    <table class="ranking-table">
"""

for _, row in rank_df.iterrows():

    r = int(row["final_rank"])

    rank_display = {
        1: "🥇",
        2: "🥈",
        3: "🥉"
    }.get(r, f"#{r}")

    model_display = pretty_model_name(row["model"])

    html += f"""
    <tr class="ranking-row">
        <td class="rank-num">{rank_display}</td>
        <td class="model-name">{model_display}</td>
        <td class="avg-rank-val">{row['avg_rank']:.2f}</td>
    </tr>
    """

html += """
    </table>
</div>
"""

display(HTML(html))

🥇,CatBoost,1.33
🥈,Gradient Boosting,2.50
🥉,XGBoost,3.17
#4,LightGBM,3.67
#5,Random Forest,4.67
#6,Extra Trees,5.67
#7,Decision Tree,7.00


## 13. Statistical Significance Testing

To determine whether the observed differences between algorithms are statistically significant, non-parametric statistical tests are applied.

Following common practice in machine learning research, the **Friedman test** is used to evaluate whether significant differences exist among the algorithms across multiple experimental conditions.

The Friedman test compares the **average ranks** of the algorithms rather than their raw performance values. This approach is appropriate when multiple algorithms are evaluated across several datasets or experimental settings.

If the Friedman test rejects the null hypothesis, it indicates that at least one algorithm performs significantly differently from the others.

To identify which specific algorithms differ significantly, the **Nemenyi post-hoc test** is applied. This test compares all pairs of algorithms using their average ranks. If the difference between the average ranks of two algorithms exceeds the **Critical Difference (CD)**, their performance difference is considered statistically significant.

In this multi-class notebook, statistical analysis is performed using **Macro F1-score**, ensuring consistency with the primary evaluation metric used throughout the experiments.

In [21]:

# CORE FUNCTION — FRIEDMAN + NEMENYI
import numpy as np
from scipy import stats
from scipy.stats import studentized_range

def compute_friedman_nemenyi(df, condition_col, metric):

    pivot = df.pivot_table(
        index=condition_col,
        columns="model",
        values=metric,
        aggfunc="mean"
    )

    pivot = pivot.dropna()

    # Friedman test
    stat, p = stats.friedmanchisquare(
        *[pivot[col].values for col in pivot.columns]
    )

    # Ranking
    ranks = pivot.rank(axis=1, ascending=False)
    avg_ranks = ranks.mean().sort_values()

    # Nemenyi CD
    k = len(pivot.columns)
    N = len(pivot)

    q_alpha = studentized_range.ppf(1 - 0.05, k, np.inf) / np.sqrt(2)
    CD = q_alpha * np.sqrt(k * (k + 1) / (6 * N))

    # Significant pairs
    significant_pairs = []
    models = avg_ranks.index.tolist()

    for i in range(len(models)):
        for j in range(i + 1, len(models)):
            diff = abs(avg_ranks[models[i]] - avg_ranks[models[j]])
            if diff > CD:
                significant_pairs.append((models[i], models[j], diff))

    return stat, p, avg_ranks, CD, significant_pairs

In [22]:
# STATISTICAL SIGNIFICANCE TEST — SIZE

size_df = results_size_multi_df.copy()

# Combine dataset + size → condition
size_df["condition"] = (
    size_df["dataset"] + "_size_" + size_df["size"].astype(str)
)

print("\n" + "="*70)
print("FRIEDMAN TEST — SIZE (MULTI-CLASS)")
print("="*70)

stat, p, avg_ranks, CD, pairs = compute_friedman_nemenyi(
    size_df,
    "condition",
    "macro_f1"
)

significance = "SIGNIFICANT" if p < 0.05 else "NOT SIGNIFICANT"

print(f"\nFriedman: stat={stat:.4f}, p={p:.6f} → {significance}")

print("\n" + "="*70)
print("NEMENYI POST-HOC — SIZE (MULTI-CLASS)")
print("="*70)

if p >= 0.05:
    print("No post-hoc analysis (Friedman not significant)")
else:

    print("\nAverage ranks:")
    for m, r in avg_ranks.items():
        print(f"{m:<18} {r:.4f}")

    print(f"\nCritical Difference (CD) = {CD:.4f}")

    if len(pairs) == 0:
        print("No significant pairwise differences")
    else:
        print("\nSignificant pairs:")
        for m1, m2, diff in pairs:
            print(f"{m1} vs {m2} (diff={diff:.4f})")


FRIEDMAN TEST — SIZE (MULTI-CLASS)

Friedman: stat=30.2143, p=0.000036 → SIGNIFICANT

NEMENYI POST-HOC — SIZE (MULTI-CLASS)

Average ranks:
CatBoost           1.1667
XGBoost            2.3333
LightGBM           3.3333
GradientBoosting   3.8333
RandomForest       4.6667
ExtraTrees         5.6667
DecisionTree       7.0000

Critical Difference (CD) = 3.6772

Significant pairs:
CatBoost vs ExtraTrees (diff=4.5000)
CatBoost vs DecisionTree (diff=5.8333)
XGBoost vs DecisionTree (diff=4.6667)


In [23]:

# STATISTICAL SIGNIFICANCE TEST — IMBALANCE

imb_df = results_imbalance_multi_df.copy()

# Combine dataset + IR → condition
imb_df["condition"] = (
    imb_df["dataset"] + "_IR_" + imb_df["imbalance_ratio"].astype(str)
)

print("\n" + "="*70)
print("FRIEDMAN TEST — IMBALANCE (MULTI-CLASS)")
print("="*70)

stat, p, avg_ranks, CD, pairs = compute_friedman_nemenyi(
    imb_df,
    "condition",
    "macro_f1"
)

significance = "SIGNIFICANT" if p < 0.05 else "NOT SIGNIFICANT"

print(f"\nFriedman: stat={stat:.4f}, p={p:.6f} → {significance}")

print("\n" + "="*70)
print("NEMENYI POST-HOC — IMBALANCE (MULTI-CLASS)")
print("="*70)

if p >= 0.05:
    print("No post-hoc analysis (Friedman not significant)")
else:

    print("\nAverage ranks:")
    for m, r in avg_ranks.items():
        print(f"{m:<18} {r:.4f}")

    print(f"\nCritical Difference (CD) = {CD:.4f}")

    if len(pairs) == 0:
        print("No significant pairwise differences")
    else:
        print("\nSignificant pairs:")
        for m1, m2, diff in pairs:
            print(f"{m1} vs {m2} (diff={diff:.4f})")


FRIEDMAN TEST — IMBALANCE (MULTI-CLASS)

Friedman: stat=28.7857, p=0.000067 → SIGNIFICANT

NEMENYI POST-HOC — IMBALANCE (MULTI-CLASS)

Average ranks:
CatBoost           1.3333
GradientBoosting   2.5000
XGBoost            3.1667
LightGBM           3.6667
RandomForest       4.6667
ExtraTrees         5.6667
DecisionTree       7.0000

Critical Difference (CD) = 3.6772

Significant pairs:
CatBoost vs ExtraTrees (diff=4.3333)
CatBoost vs DecisionTree (diff=5.6667)
GradientBoosting vs DecisionTree (diff=4.5000)
XGBoost vs DecisionTree (diff=3.8333)
